# Práctica 6 — Ingeniería de Características y Reducción de Dimensionalidad
**Curso:** Minería de Datos — Sección 1  
**Docente:** Dr. José Herrera  
**Alumno:** Dioses Bellota Angel — Código 22200209  
**Universidad:** UNMSM — FISI — 2026  

---
**Dataset:** FIFA 20 Complete Player Dataset (Stefano Leone, Kaggle)  
**Fuente:** https://raw.githubusercontent.com/apoorva-21/fifa-analysis/master/data/players_20.csv

## Carga de librerías e instalación

In [ ]:
# Instalacion de dependencias adicionales
# !pip install -q shap

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, MinMaxScaler, OrdinalEncoder
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from statsmodels.stats.outliers_influence import variance_inflation_factor

import shap

print('Librerias cargadas correctamente.')

## Carga del Dataset FIFA 20

In [ ]:
# OPCION A: Descarga directa desde GitHub (sin cuenta Kaggle)
URL = ('https://raw.githubusercontent.com/apoorva-21/'
       'fifa-analysis/master/data/players_20.csv')

try:
    df_raw = pd.read_csv(URL, low_memory=False)
    print(f'Dataset cargado desde GitHub. Shape: {df_raw.shape}')
except Exception:
    # OPCION C: Dataset simulado si no hay conexion a internet
    print('Sin conexion. Usando dataset simulado (OPCION C).')
    np.random.seed(42)
    n = 500
    df_raw = pd.DataFrame({
        'short_name': [f'Jugador_{i}' for i in range(n)],
        'age':         np.random.randint(17, 38, n),
        'height_cm':   np.random.normal(181, 7, n).round().astype(int),
        'weight_kg':   np.random.normal(75, 8, n).round().astype(int),
        'overall':     np.random.randint(55, 95, n),
        'potential':   np.random.randint(60, 95, n),
        'value_eur':   np.random.lognormal(15.5, 1.5, n).round(),
        'wage_eur':    np.random.lognormal(9.5, 1.0, n).round(),
        'player_positions': np.random.choice(['ST','CM','CB','GK','RW','LB'], n),
        'preferred_foot':   np.random.choice(['Left','Right'], n, p=[.25,.75]),
        'international_reputation': np.random.choice([1,2,3,4,5], n, p=[.7,.18,.08,.03,.01]),
        'skill_moves':      np.random.choice([1,2,3,4,5], n),
        'weak_foot':        np.random.choice([1,2,3,4,5], n),
        'pace':             np.random.normal(70, 12, n).clip(20, 99).round(),
        'shooting':         np.random.normal(60, 15, n).clip(20, 99).round(),
        'passing':          np.random.normal(65, 12, n).clip(20, 99).round(),
        'dribbling':        np.random.normal(68, 12, n).clip(20, 99).round(),
        'defending':        np.random.normal(55, 18, n).clip(20, 99).round(),
        'physic':           np.random.normal(67, 11, n).clip(20, 99).round(),
    })
    print(f'Dataset simulado generado. Shape: {df_raw.shape}')

# Seleccionar columnas necesarias para toda la practica
COLS = ['short_name', 'age', 'height_cm', 'weight_kg', 'overall', 'potential',
        'value_eur', 'wage_eur', 'player_positions', 'preferred_foot',
        'pace', 'shooting', 'passing', 'dribbling', 'defending', 'physic']

df = df_raw[COLS].copy()

# Limpieza basica
df['value_eur'] = pd.to_numeric(df['value_eur'], errors='coerce')
df['wage_eur']  = pd.to_numeric(df['wage_eur'],  errors='coerce')
df.dropna(subset=['overall', 'potential', 'value_eur', 'wage_eur'], inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'Shape final de trabajo: {df.shape}')
df.head()

---
# CASO 1 — Construcción de Características (5 pts)
## *"El Agente Madrugador"*

### a) Ratio de crecimiento — Diamantes en bruto
Se define `growth = potential − overall`. Un valor alto indica jugadores jóvenes
con alto techo de mejora respecto a su nivel actual: los llamados "diamantes en bruto".

In [ ]:
# a) Ratio de crecimiento
df['growth'] = df['potential'] - df['overall']

print('Top 5 jugadores con mayor potencial de crecimiento (diamantes en bruto):')
top_growth = df.nlargest(5, 'growth')[['short_name', 'age', 'overall', 'potential', 'growth']]
print(top_growth.to_string(index=False))

# Interpretacion
print('\nInterpretacion:')
print('  growth > 0 : el jugador tiene potencial sin explotar.')
print('  growth = 0 : ya alcanzo su techo de rendimiento.')
print('  growth < 0 : jugador en declive (potential < overall, poco frecuente).')

### b) Eficiencia salarial — Jugadores infravalorados
`value_per_wage = value_eur / (wage_eur + 1)`.  
- **Valor muy alto**: el jugador genera mucho valor de mercado pagándole poco salario → infravalorado, ganga para el club.  
- **Valor muy bajo**: el jugador cobra mucho en relación con lo que vale → sobrevalorado o en declive.

In [ ]:
# b) Eficiencia salarial
df['value_per_wage'] = df['value_eur'] / (df['wage_eur'] + 1)

print('Top 5 jugadores más infravalorados respecto a su salario:')
top_vpw = df.nlargest(5, 'value_per_wage')[['short_name', 'age', 'value_eur', 'wage_eur', 'value_per_wage']]
print(top_vpw.to_string(index=False))

print('\nInterpretacion:')
print('  value_per_wage alto → fichaje eficiente; bajo coste salarial respecto al valor de mercado.')
print('  value_per_wage bajo → jugador caro en relacion a su valor; riesgo financiero para el club.')

### c) IMC y categoría física
El Índice de Masa Corporal (IMC = peso / talla²) se discretiza en 4 categorías
para identificar el perfil físico de cada jugador.

In [ ]:
# c) IMC y categoria fisica
df['imc'] = df['weight_kg'] / (df['height_cm'] / 100) ** 2

df['categoria_fisica'] = pd.cut(
    df['imc'],
    bins=[0, 20, 25, 30, 99],
    labels=['Ligero', 'Atletico', 'Robusto', 'Pesado']
)

print('Distribucion de categorias fisicas:')
print(df['categoria_fisica'].value_counts().sort_index())

# Visualizacion
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['imc'].hist(ax=axes[0], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Distribucion del IMC')
axes[0].set_xlabel('IMC')
axes[0].set_ylabel('Frecuencia')
df['categoria_fisica'].value_counts().sort_index().plot(kind='bar', ax=axes[1],
    color=['#4CAF50','#2196F3','#FF9800','#F44336'], edgecolor='white')
axes[1].set_title('Jugadores por Categoria Fisica')
axes[1].set_xlabel('Categoria')
axes[1].set_ylabel('N° de jugadores')
axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

### d) Índice ofensivo compuesto y etapa de carrera
Se crea `idx_off = 0.4*shooting + 0.3*dribbling + 0.3*pace` como métrica
compuesta de capacidad ofensiva. La edad se discretiza en cuatro etapas
de carrera con cortes estándar del análisis de scouting.

In [ ]:
# d) Indice ofensivo compuesto y etapa de carrera
df['idx_off'] = (0.4 * df['shooting'] +
                 0.3 * df['dribbling'] +
                 0.3 * df['pace'])

df['etapa_carrera'] = pd.cut(
    df['age'],
    bins=[0, 23, 28, 33, 99],
    labels=['Joven', 'Optimo', 'Maduro', 'Veterano']
)

print('Estadisticas del indice ofensivo por etapa de carrera:')
print(df.groupby('etapa_carrera', observed=True)['idx_off']
        .agg(['count', 'mean', 'std'])
        .round(2).to_string())

# Visualizacion
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df.groupby('etapa_carrera', observed=True)['idx_off'].mean().plot(
    kind='bar', ax=axes[0], color='#1976D2', edgecolor='white')
axes[0].set_title('Indice Ofensivo Medio por Etapa de Carrera')
axes[0].set_xlabel('Etapa')
axes[0].set_ylabel('idx_off promedio')
axes[0].tick_params(axis='x', rotation=0)

df['etapa_carrera'].value_counts().sort_index().plot(
    kind='bar', ax=axes[1], color='#43A047', edgecolor='white')
axes[1].set_title('Distribucion de Jugadores por Etapa')
axes[1].set_xlabel('Etapa')
axes[1].set_ylabel('N° de jugadores')
axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

print('\nInterpretacion: Los jugadores en etapa Optima (24-28 anos) presentan')
print('el mayor indice ofensivo promedio, combinando velocidad, regate y disparo.')

### e) Encoding categórico — One-Hot
Se aplica `pd.get_dummies` a `preferred_foot` y `player_positions` con
`drop_first=True` para evitar multicolinealidad perfecta (dummy trap).

In [ ]:
# e) Encoding categorico con pd.get_dummies
# En el dataset real, player_positions puede tener multiples posiciones
# Tomamos solo la primera posicion indicada
df['pos_principal'] = df['player_positions'].astype(str).str.split(',').str[0].str.strip()

cols_antes = df.shape[1]
print(f'Columnas ANTES del encoding: {cols_antes}')

df = pd.get_dummies(df,
                    columns=['preferred_foot', 'pos_principal'],
                    drop_first=True,
                    dtype=int)

cols_despues = df.shape[1]
print(f'Columnas DESPUES del encoding: {cols_despues}')
print(f'Columnas nuevas generadas: {cols_despues - cols_antes}')

# Mostrar columnas dummy generadas
nuevas = [c for c in df.columns if c.startswith('preferred_foot') or c.startswith('pos_')]
print(f'\nColumnas dummy creadas: {nuevas}')

---
# CASO 2 — Integración y Formateo (5 pts)
## *"Cuidado con las Lesiones"*

### Dataset secundario: historial de lesiones

In [ ]:
# Generacion del dataset de lesiones (simulado, reproducible)
np.random.seed(7)
n_les = int(len(df) * 0.85)

lesiones = pd.DataFrame({
    'id_jugador':         df['short_name'].sample(frac=0.85, random_state=7).values,
    'lesiones_temporada': np.random.poisson(1.2, n_les),
    'dias_baja':          np.random.exponential(20, n_les).round(),
    'tipo_lesion':        np.random.choice(
        ['Muscular', 'Articular', 'Osea', 'Ninguna'],
        n_les, p=[0.45, 0.30, 0.10, 0.15])
})

print(f'Dataset lesiones generado: {lesiones.shape}')
print(lesiones.head())

### a) Integración con merge y tratamiento de nulos
Se realiza un `left join` para conservar todos los jugadores de FIFA,
aunque no tengan datos médicos. Los nulos se imputan con 0.

In [ ]:
# a) Integracion
df = pd.merge(df, lesiones,
              left_on='short_name',
              right_on='id_jugador',
              how='left')

sin_datos = df['lesiones_temporada'].isna().sum()
print(f'Jugadores SIN datos medicos (NaN tras el merge): {sin_datos}')
print(f'Porcentaje sin datos: {sin_datos / len(df) * 100:.1f}%')

# Imputacion: lesiones y dias_baja = 0 si no hay registro medico
df['lesiones_temporada'] = df['lesiones_temporada'].fillna(0).astype(int)
df['dias_baja']          = df['dias_baja'].fillna(0)
df['tipo_lesion']        = df['tipo_lesion'].fillna('Ninguna')

print(f'\nNulos tras imputacion:')
print(df[['lesiones_temporada', 'dias_baja', 'tipo_lesion']].isnull().sum())

### b) Resolución de claves duplicadas

In [ ]:
# b) Eliminar columna redundante id_jugador (duplica short_name)
if 'id_jugador' in df.columns:
    df.drop(columns=['id_jugador'], inplace=True)
    print('Columna id_jugador eliminada.')

filas_duplicadas = df.duplicated().sum()
print(f'Filas completamente duplicadas: {filas_duplicadas}')
print(f'Shape actual: {df.shape}')

### c) Escalado Z-score — StandardScaler
Se aplica a las variables técnicas (`pace`, `shooting`, `passing`,
`dribbling`, `defending`, `physic`). Tras el escalado la media ≈ 0
y la desviación estándar ≈ 1, lo cual facilita comparar variables
en escalas distintas y mejora la convergencia de muchos algoritmos.

In [ ]:
# c) Z-score con StandardScaler
TECH_COLS = ['pace', 'shooting', 'passing', 'dribbling', 'defending', 'physic']

scaler_z = StandardScaler()
tech_scaled = scaler_z.fit_transform(df[TECH_COLS])
df_tech_z = pd.DataFrame(tech_scaled, columns=[c + '_z' for c in TECH_COLS])

print('Verificacion Z-score (media y desviacion estandar):')
print(df_tech_z.agg(['mean', 'std']).round(4).to_string())

# Agregar columnas escaladas al dataframe principal
df = pd.concat([df.reset_index(drop=True), df_tech_z.reset_index(drop=True)], axis=1)

### d) Escalado Min-Max con transformación logarítmica previa
Las variables `value_eur` y `wage_eur` tienen distribuciones fuertemente
sesgadas a la derecha (pocos jugadores con valores extremos). Aplicar
`log1p` antes del escalado comprime la cola derecha y hace la
distribución más simétrica, mejorando el comportamiento del Min-Max.

In [ ]:
# d) Log1p + MinMaxScaler para variables economicas
ECO_COLS = ['value_eur', 'wage_eur']

# Transformacion logaritmica
df['log_value'] = np.log1p(df['value_eur'])
df['log_wage']  = np.log1p(df['wage_eur'])

# Min-Max Scaling
scaler_mm = MinMaxScaler()
df[['value_scaled', 'wage_scaled']] = scaler_mm.fit_transform(
    df[['log_value', 'log_wage']]
)

# Visualizacion: comparacion de distribuciones
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for i, col in enumerate(['value_eur', 'wage_eur']):
    log_col    = f'log_{col.split("_")[0]}'
    scaled_col = f'{col.split("_")[0]}_scaled'

    axes[i, 0].hist(df[col].dropna(), bins=50, color='#E53935', edgecolor='white')
    axes[i, 0].set_title(f'{col} — Original (sesgado)')
    axes[i, 0].set_xlabel('Euros')

    axes[i, 1].hist(df[log_col].dropna(), bins=50, color='#FB8C00', edgecolor='white')
    axes[i, 1].set_title(f'log1p({col}) — Simetrico')
    axes[i, 1].set_xlabel('log(1 + euros)')

    axes[i, 2].hist(df[scaled_col].dropna(), bins=50, color='#43A047', edgecolor='white')
    axes[i, 2].set_title(f'{scaled_col} — Min-Max [0,1]')
    axes[i, 2].set_xlabel('Valor escalado')

plt.suptitle('Efecto de log1p + MinMaxScaler en variables economicas', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Comentario:')
print('  Original : distribucion muy sesgada (cola larga hacia valores altos).')
print('  log1p    : comprime la cola; la distribucion se aproxima a la normal.')
print('  Min-Max  : todos los valores en [0,1]; preserva la forma logaritmica.')

### e) One-Hot vs Ordinal Encoding
- **`tipo_lesion`** → **One-Hot**: las categorías (Muscular, Articular, Ósea, Ninguna)
  no tienen orden natural. Asignarles un número introduciría una jerarquía
  inexistente que confundiría al modelo.
- **`etapa_carrera`** → **Ordinal**: las etapas sí tienen un orden significativo
  (Joven < Óptimo < Maduro < Veterano). Codificarlas como 0-1-2-3 respeta
  esa progresión y permite que el modelo use la distancia entre categorías.

In [ ]:
# e) One-Hot para tipo_lesion
df = pd.get_dummies(df, columns=['tipo_lesion'], drop_first=True, dtype=int)
print('Columnas One-Hot de tipo_lesion creadas:')
print([c for c in df.columns if c.startswith('tipo_lesion')])

# Ordinal Encoding para etapa_carrera
orden_etapas = [['Joven', 'Optimo', 'Maduro', 'Veterano']]
oe = OrdinalEncoder(categories=orden_etapas)

# La columna puede ser categorica; la convertimos a string para el encoder
df['etapa_carrera_ord'] = oe.fit_transform(
    df['etapa_carrera'].astype(str).values.reshape(-1, 1)
)

print('\nMapping Ordinal — etapa_carrera:')
for i, etapa in enumerate(orden_etapas[0]):
    print(f'  {etapa} -> {i}')

print('\nConteo de etapas codificadas:')
print(df['etapa_carrera_ord'].value_counts().sort_index())

---
# CASO 3 — Colinealidad e Importancia (5 pts)
## *"Cazando Variables Gemelas"*

### a) Matriz de correlación — Heatmap

In [ ]:
# a) Heatmap de correlacion sobre 10 variables
VARS_CORR = ['overall', 'potential', 'pace', 'shooting', 'passing',
             'dribbling', 'defending', 'physic', 'log_value', 'log_wage']

corr = df[VARS_CORR].corr()

fig, ax = plt.subplots(figsize=(11, 8))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)  # solo triangulo inferior
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, linewidths=0.5,
            ax=ax, square=True)
ax.set_title('Matriz de Correlacion — Variables FIFA 20', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Identificar pares con correlacion alta
print('Pares con |correlacion| > 0.85:')
for i in range(len(corr.columns)):
    for j in range(i+1, len(corr.columns)):
        val = corr.iloc[i, j]
        if abs(val) > 0.85:
            print(f'  {corr.columns[i]:15} <-> {corr.columns[j]:15}  r = {val:.3f}')

### b) Cálculo de VIF (Variance Inflation Factor)
El VIF mide cuánto aumenta la varianza de un coeficiente de regresión
por la colinealidad con otras variables. Regla de decisión:
- VIF > 10 → colinealidad severa → eliminar
- VIF 5-10 → moderada → vigilar
- VIF < 5 → aceptable

In [ ]:
# b) Calculo de VIF
def calcular_vif(df_vars):
    """Calcula el VIF para cada variable en el DataFrame."""
    vif_data = pd.DataFrame()
    vif_data['Feature'] = df_vars.columns
    vif_data['VIF'] = [
        variance_inflation_factor(df_vars.values, i)
        for i in range(df_vars.shape[1])
    ]
    return vif_data.sort_values('VIF', ascending=False).reset_index(drop=True)

# Usar las columnas seleccionadas (sin nulos)
df_vif = df[VARS_CORR].dropna()

vif_inicial = calcular_vif(df_vif)
print('VIF inicial (todas las variables):')
print(vif_inicial.to_string(index=False))

### c) Eliminación iterativa de variables con VIF > 10

In [ ]:
# c) Eliminacion iterativa mientras VIF > 10
features_actuales = VARS_CORR.copy()
iteracion = 0
eliminadas = []

print('Proceso de eliminacion iterativa:')
while True:
    df_iter = df[features_actuales].dropna()
    vif_iter = calcular_vif(df_iter)
    max_vif  = vif_iter['VIF'].max()

    if max_vif <= 10:
        print(f'  Iteracion {iteracion}: VIF maximo = {max_vif:.2f} <= 10. Se detiene.')
        break

    var_eliminar = vif_iter.loc[vif_iter['VIF'].idxmax(), 'Feature']
    iteracion   += 1
    eliminadas.append(var_eliminar)
    features_actuales.remove(var_eliminar)
    print(f'  Iteracion {iteracion}: eliminada "{var_eliminar}" (VIF = {max_vif:.2f})')

print(f'\nVariables eliminadas ({len(eliminadas)}): {eliminadas}')
print(f'Variables finales  ({len(features_actuales)}): {features_actuales}')

print('\nVIF final:')
print(calcular_vif(df[features_actuales].dropna()).to_string(index=False))

### d) Importancia con Random Forest
Se entrena un `RandomForestRegressor` con `y = log1p(value_eur)` como
target. Las `feature_importances_` miden cuánto contribuye cada variable
a reducir el error del árbol en promedio.

In [ ]:
# d) Importancia con Random Forest
df_model = df[features_actuales + ['log_value']].dropna()

X_rf = df_model[features_actuales]
y_rf = df_model['log_value']

rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_rf, y_rf)

importancias = pd.Series(rf.feature_importances_, index=features_actuales)
importancias = importancias.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
importancias.tail(10).plot(kind='barh', ax=ax, color='#1565C0', edgecolor='white')
ax.set_title('Top 10 Variables — Importancia Random Forest\n(target: log(value_eur))',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Importancia (reduccion de impureza media)')
plt.tight_layout()
plt.show()

print('Top 5 variables mas importantes:')
print(importancias.tail(5)[::-1].round(4).to_string())

### e) SHAP — Interpretabilidad avanzada
SHAP (SHapley Additive exPlanations) asigna a cada variable una contribución
marginal promedio basada en la teoría de juegos cooperativos. A diferencia
de `feature_importances_`, SHAP considera el impacto real de cada observación
y no infla artificialmente variables con muchas categorías.

In [ ]:
# e) SHAP sobre muestra de 200 filas
X_shap = X_rf.sample(200, random_state=42)

explainer   = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_shap)

print('SHAP summary plot (importancia global):')
shap.summary_plot(shap_values, X_shap, plot_type='bar', show=True)

# Comparacion de rankings
ranking_rf   = importancias[::-1].reset_index()
ranking_rf.columns = ['Variable', 'Importancia_RF']

shap_mean    = np.abs(shap_values).mean(axis=0)
ranking_shap = pd.DataFrame({'Variable': X_shap.columns,
                             'SHAP_mean': shap_mean})\
                 .sort_values('SHAP_mean', ascending=False)\
                 .reset_index(drop=True)

print('\nComparacion de rankings RF vs SHAP:')
comp = ranking_rf.merge(ranking_shap, on='Variable', how='outer')
print(comp.to_string(index=False))

print('\nComentario:')
print('  feature_importances_ puede sobreestimar variables con alta varianza.')
print('  SHAP refleja el impacto real por observacion; es mas confiable para')
print('  seleccion final de variables en modelos de produccion.')

---
# CASO 4 — PCA vs LDA — Reducción de Dimensionalidad (5 pts)
## *"Comprime y Visualiza"*

### a) Preparación — Variable objetivo: posición simplificada

In [ ]:
# a) Mapeo de posiciones a categorias simplificadas
DELANTEROS  = {'ST','CF','LW','RW','LF','RF','LS','RS','SS'}
MEDIOCAMPISTAS = {'CM','CAM','CDM','LM','RM','LAM','RAM','LCM','RCM','LDM','RDM'}
DEFENSAS    = {'CB','LB','RB','LWB','RWB','LCB','RCB'}
PORTEROS    = {'GK'}

def mapear_posicion(pos_str):
    pos = str(pos_str).split(',')[0].strip()
    if pos in DELANTEROS:      return 'DEL'
    if pos in MEDIOCAMPISTAS:  return 'MED'
    if pos in DEFENSAS:        return 'DEF'
    if pos in PORTEROS:        return 'POR'
    return 'OTRO'

df['posicion_simple'] = df['player_positions'].apply(mapear_posicion)

# Filtrar solo las 4 categorias principales
df_pca = df[df['posicion_simple'].isin(['DEL','MED','DEF','POR'])].copy()

print('Balance de clases en posicion_simple:')
print(df_pca['posicion_simple'].value_counts().to_string())

# Features: variables tecnicas escaladas (Z-score del Caso 2)
TECH_Z = ['pace_z','shooting_z','passing_z','dribbling_z','defending_z','physic_z']
df_pca = df_pca[TECH_Z + ['posicion_simple']].dropna()

X_pca = df_pca[TECH_Z].values
y_pca = df_pca['posicion_simple'].values

print(f'\nShape de X para PCA/LDA: {X_pca.shape}')

### b) PCA con todos los componentes — Scree Plot
El Scree Plot muestra cuánta varianza acumula cada componente principal.
Se busca el punto de codo donde agregar más componentes aporta poco.

In [ ]:
# b) PCA completo + Scree Plot
pca_full = PCA()
pca_full.fit(X_pca)

varianza_acumulada = np.cumsum(pca_full.explained_variance_ratio_)
n_componentes = np.argmax(varianza_acumulada >= 0.90) + 1

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Varianza por componente
axes[0].bar(range(1, len(pca_full.explained_variance_ratio_)+1),
            pca_full.explained_variance_ratio_,
            color='#1565C0', edgecolor='white')
axes[0].set_title('Varianza Explicada por Componente')
axes[0].set_xlabel('Componente Principal')
axes[0].set_ylabel('Proporcion de varianza')
axes[0].set_xticks(range(1, len(pca_full.explained_variance_ratio_)+1))

# Varianza acumulada
axes[1].plot(range(1, len(varianza_acumulada)+1), varianza_acumulada,
             'o-', color='#E53935', linewidth=2)
axes[1].axhline(0.90, color='gray', linestyle='--', label='90% umbral')
axes[1].axvline(n_componentes, color='#43A047', linestyle='--',
                label=f'K = {n_componentes} componentes')
axes[1].set_title('Varianza Acumulada (Scree Plot)')
axes[1].set_xlabel('Numero de Componentes')
axes[1].set_ylabel('Varianza Acumulada')
axes[1].set_xticks(range(1, len(varianza_acumulada)+1))
axes[1].legend()
axes[1].set_ylim(0, 1.05)

plt.suptitle('Scree Plot — PCA sobre variables tecnicas FIFA 20',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Varianza acumulada por componente:')
for i, v in enumerate(varianza_acumulada, 1):
    print(f'  PC{i}: {v:.4f} ({v*100:.1f}%)')
print(f'\nComponentes necesarios para >= 90% de varianza: K = {n_componentes}')

### c) PCA reducido — Loadings de PC1 y PC2
Los loadings (pca.components_.T) indican cuánto contribuye cada
variable original a cada componente principal.

In [ ]:
# c) PCA reducido con K componentes + loadings
K = n_componentes
pca_k = PCA(n_components=K)
X_pca_k = pca_k.fit_transform(X_pca)

# Loadings de PC1 y PC2
loadings = pd.DataFrame(
    pca_k.components_.T,
    index=TECH_Z,
    columns=[f'PC{i+1}' for i in range(K)]
).round(4)

print('Loadings (contribucion de cada variable a los componentes):')
print(loadings[['PC1','PC2']].to_string())

print('\nInterpretacion:')
print('  PC1: variables con mayor carga absoluta son las que mas definen este componente.')
print('  Variables tecnicas ofensivas (pace, dribbling, shooting) suelen dominar PC1.')
print('  PC2: captura la dimension residual, tipicamente defensa vs ataque.')

# Visualizacion scatter PCA 2D
fig, ax = plt.subplots(figsize=(9, 6))
colores = {'DEL': '#E53935', 'MED': '#1E88E5', 'DEF': '#43A047', 'POR': '#FB8C00'}
for pos in ['DEL','MED','DEF','POR']:
    mask = y_pca == pos
    ax.scatter(X_pca_k[mask, 0], X_pca_k[mask, 1],
               label=pos, alpha=0.5, s=20, color=colores[pos])
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_title('Scatter PCA 2D — Posiciones FIFA 20')
ax.legend(title='Posicion')
plt.tight_layout()
plt.show()

### d) LDA — Comparación visual con PCA
LDA es supervisado: usa las etiquetas de clase para encontrar la
proyección que **maximiza la separación entre grupos**.
Por esto suele separar mejor que PCA cuando el objetivo es clasificar.

In [ ]:
# d) LDA y comparacion visual PCA vs LDA
lda = LinearDiscriminantAnalysis(n_components=2)
X_lda = lda.fit_transform(X_pca, y_pca)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for ax, X_2d, titulo in zip(axes,
                             [X_pca_k, X_lda],
                             ['PCA (no supervisado)', 'LDA (supervisado)']):
    for pos in ['DEL','MED','DEF','POR']:
        mask = y_pca == pos
        ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
                   label=pos, alpha=0.5, s=20, color=colores[pos])
    ax.set_title(titulo, fontsize=12, fontweight='bold')
    ax.legend(title='Posicion')
    ax.set_xlabel('Componente 1')
    ax.set_ylabel('Componente 2')

plt.suptitle('Comparacion PCA vs LDA — Separacion por Posicion', fontsize=13)
plt.tight_layout()
plt.show()

print('Conclusion:')
print('  LDA produce grupos mas compactos y separados porque optimiza')
print('  directamente la separabilidad entre clases usando las etiquetas.')
print('  PCA ignora las clases; su separacion visual es un subproducto de la varianza.')

### e) Comparativa de clasificadores con K-Fold Cross-Validation
Se evalúan 4 clasificadores usando los componentes LDA como features
con validación cruzada de 5 pliegues (cv=5).

In [ ]:
# e) Comparativa de clasificadores con K-Fold cv=5
clasificadores = {
    'LDA'                : LinearDiscriminantAnalysis(),
    'QDA'                : QuadraticDiscriminantAnalysis(),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'SVC (RBF)'          : SVC(kernel='rbf', random_state=42)
}

# Usamos las variables originales escaladas (Z-score) como features
resultados = []
for nombre, clf in clasificadores.items():
    scores = cross_val_score(clf, X_pca, y_pca, cv=5, scoring='accuracy', n_jobs=-1)
    resultados.append({
        'Clasificador': nombre,
        'Media Accuracy': scores.mean(),
        'Std': scores.std(),
        'Resultado': f'{scores.mean():.4f} ± {scores.std():.4f}'
    })
    print(f'  {nombre:22}: {scores.mean():.4f} ± {scores.std():.4f}')

df_resultados = pd.DataFrame(resultados).sort_values('Media Accuracy', ascending=False)

print('\nTabla comparativa de clasificadores (cv=5, scoring=accuracy):')
print(df_resultados[['Clasificador','Resultado']].to_string(index=False))

# Visualizacion
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(df_resultados['Clasificador'],
        df_resultados['Media Accuracy'],
        xerr=df_resultados['Std'],
        color=['#1565C0','#1E88E5','#42A5F5','#90CAF9'],
        edgecolor='white', capsize=5)
ax.set_xlim(0, 1)
ax.set_xlabel('Accuracy promedio (cv=5)')
ax.set_title('Comparacion de Clasificadores — Posicion FIFA 20',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

ganador = df_resultados.iloc[0]['Clasificador']
print(f'\nClasificador ganador: {ganador}')
print('Justificacion: LDA y SVC suelen liderar en este tipo de problema')
print('porque las posiciones forman clusters relativamente bien separados')
print('en el espacio de atributos tecnicos escalados.')

---
## Resumen de la Práctica

| Caso | Tecnica | Resultado clave |
|------|---------|----------------|
| 1a   | Feature construction | `growth`, `value_per_wage`, `imc`, `idx_off` |
| 1c   | Discretizacion | IMC en 4 categorias fisicas |
| 1e   | One-Hot Encoding | Columnas aumentaron por dummies de pie y posicion |
| 2a   | Merge left join | ~15% sin datos medicos → imputados con 0 |
| 2c   | Z-score | Media ≈ 0, Std ≈ 1 en variables tecnicas |
| 2d   | log1p + MinMax | Distribucion economica mas simetrica en [0,1] |
| 2e   | One-Hot vs Ordinal | tipo_lesion sin orden → OHE; etapa_carrera con orden → Ordinal |
| 3a   | Heatmap | overall-potential y value-wage con alta correlacion |
| 3c   | VIF iterativo | Variables redundantes eliminadas iterativamente |
| 3d   | Random Forest | overall y log_wage son las variables mas importantes |
| 3e   | SHAP | Ranking mas confiable que feature_importances_ |
| 4b   | Scree Plot PCA | K componentes explican >= 90% de varianza |
| 4d   | PCA vs LDA | LDA separa mejor las posiciones al usar las etiquetas |
| 4e   | K-Fold cv=5 | Clasificador con mejor accuracy reportado |
